# LAB03 — Ensambles, reducción dimensional y Green AI

Este notebook reutiliza el dataset, el target y la partición ya congelados
en `02_dataset_publico.ipynb` (LAB02) y compara varios modelos bajo un
mismo protocolo: mismo entrenamiento/prueba, misma métrica principal.

El estado de aprobación docente del dataset se documenta en
`docs/ficha_dataset.md`; este notebook no vuelve a discutir esa decisión,
solo la reutiliza tal cual quedó definida.

In [ ]:
import sys
from pathlib import Path

# Asegura que 'src' esté en sys.path sin importar el cwd del kernel
_root = Path.cwd()
while not (_root / "src").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root / "src"))

## Paso 1: Congelar el protocolo

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Protocolo congelado del LAB02: mismo dataset, mismo target, misma
# columna excluida por fuga de informacion, mismo ColumnTransformer y
# misma particion (random_state=42, test_size=0.20, stratify=y). Nada de
# esto cambia en el LAB03 -- solo se comparan modelos sobre el mismo
# terreno.
df = pd.read_csv(_root / "data/raw/dataset.csv")

TARGET = "categoria_robo"
DROP_COLUMNS = ["tipo_detalle"]

X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # sparse_output=False (diferencia respecto al LAB02): mas adelante en
    # este notebook se usan HistGradientBoostingClassifier y PCA, que
    # necesitan entrada densa. No cambia el resultado del
    # ColumnTransformer, solo el formato de salida.
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

PRIMARY_METRIC = "f1_macro"
PRIORITY_CLASS = "Armas de Fuego"  # la de error mas costoso, definida en el LAB02

print("Filas train/test:", Xtr.shape[0], Xte.shape[0])
print("Metrica principal:", PRIMARY_METRIC)
print("Clase prioritaria:", PRIORITY_CLASS)

## Paso 2: Crear el catálogo de modelos

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# Catalogo de modelos a comparar. Todos comparten el mismo 'preprocess'
# (protocolo congelado en el paso anterior) para que la comparacion sea
# justa -- la unica variable es el modelo.
#
# A diferencia del SVM del LAB02, aqui no se activa probability=True: no
# se necesita para las metricas de este laboratorio y activarlo hace el
# entrenamiento notablemente mas lento (coherente con el enfoque Green AI
# de este LAB03: no pagar un costo computacional que no se va a usar).
models = {
    "logistic": Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000, random_state=42))]),
    "svm_c1": Pipeline([("prep", preprocess), ("model", SVC(C=1, gamma="scale", random_state=42))]),
    "svm_c10": Pipeline([("prep", preprocess), ("model", SVC(C=10, gamma="scale", random_state=42))]),
    "rf_100": Pipeline([("prep", preprocess), ("model", RandomForestClassifier(n_estimators=100, random_state=42))]),
    "rf_300": Pipeline([("prep", preprocess), ("model", RandomForestClassifier(n_estimators=300, random_state=42))]),
    "boost": Pipeline([("prep", preprocess), ("model", HistGradientBoostingClassifier(random_state=42))]),
}
print(list(models.keys()))

## Paso 3: Medir tres veces (tiempo y tamaño)

In [ ]:
import time
import joblib
from sklearn.metrics import f1_score

REPORTS_DIR = _root / "reports"
MODELS_DIR = REPORTS_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for name, pipe in models.items():
    fit_times = []
    for _ in range(3):
        t0 = time.perf_counter()
        pipe.fit(Xtr, ytr)
        fit_times.append(time.perf_counter() - t0)
    fit_median_s = sorted(fit_times)[1]  # mediana de 3 repeticiones

    t0 = time.perf_counter()
    pred = pipe.predict(Xte)
    predict_ms = (time.perf_counter() - t0) * 1000

    f1 = f1_score(yte, pred, average="macro")

    model_path = MODELS_DIR / f"{name}.joblib"
    joblib.dump(pipe, model_path)
    size_kb = model_path.stat().st_size / 1024

    rows.append({
        "model": name,
        "f1_macro": f1,
        "fit_median_s": fit_median_s,
        "predict_ms": predict_ms,
        "size_kb": size_kb,
    })
    print(f"{name}: f1={f1:.4f}  fit={fit_median_s:.3f}s  predict={predict_ms:.2f}ms  size={size_kb:.1f}KB")

results = pd.DataFrame(rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)
display(results)

results.to_csv(REPORTS_DIR / "green_ai_results.csv", index=False)
print("Guardado:", REPORTS_DIR / "green_ai_results.csv")

## Paso 4: Comparar con y sin PCA

In [ ]:
from sklearn.decomposition import PCA

# PCA solo acepta entrada numerica y densa. Como X mezcla columnas
# numericas (anio, cantidad_denuncias) y categoricas (mes, provincia),
# se aplica PCA unicamente al bloque numerico -- tal como sugiere el
# manual para datasets con columnas mixtas -- y las categoricas se dejan
# con el mismo OneHotEncoder de siempre.
#
# Nota importante sobre este dataset en particular: el bloque numerico
# solo tiene 2 columnas, asi que PCA(n_components=.95) muy probablemente
# retiene ambas componentes. Aqui no hay una reduccion real de
# dimensionalidad que demostrar -- lo que se deja documentado es el
# patron correcto para armar un pipeline de PCA sobre columnas mixtas.
num_pca_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=.95, random_state=42)),
])
preprocess_pca = ColumnTransformer([
    ("num", num_pca_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])
svm_pca = Pipeline([("prep", preprocess_pca), ("model", SVC(C=1, gamma="scale", random_state=42))])

t0 = time.perf_counter()
svm_pca.fit(Xtr, ytr)
fit_s_pca = time.perf_counter() - t0

t0 = time.perf_counter()
pred_pca = svm_pca.predict(Xte)
predict_ms_pca = (time.perf_counter() - t0) * 1000

f1_pca = f1_score(yte, pred_pca, average="macro")

model_path_pca = MODELS_DIR / "svm_c1_pca.joblib"
joblib.dump(svm_pca, model_path_pca)
size_kb_pca = model_path_pca.stat().st_size / 1024

n_componentes = svm_pca.named_steps["prep"].named_transformers_["num"].named_steps["pca"].n_components_
print(f"PCA retuvo {n_componentes} de {len(num_cols)} columnas numericas originales")
print(f"F1-macro con PCA: {f1_pca:.4f}  |  sin PCA (svm_c1): ver tabla 'results' del paso anterior")
print(f"Tiempo de entrenamiento con PCA: {fit_s_pca:.4f}s")

## Paso 5: Dos mapas t-SNE (semillas 42 y 7)

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

# Vista solo numerica, imputada y escalada (t-SNE espera coordenadas
# continuas, no columnas dispersas de categorias one-hot).
X_num = X.select_dtypes(include="number")
imputer_tsne = SimpleImputer(strategy="median")
scaler_tsne = StandardScaler()
X_num_scaled = scaler_tsne.fit_transform(imputer_tsne.fit_transform(X_num))

# Muestra de hasta 1000 filas para que t-SNE corra en tiempo razonable.
rng = np.random.default_rng(42)
n_sample = min(1000, X_num_scaled.shape[0])
sample_idx = rng.choice(X_num_scaled.shape[0], size=n_sample, replace=False)
X_sample = X_num_scaled[sample_idx]
y_sample = y.iloc[sample_idx].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, seed in zip(axes, [42, 7]):
    emb = TSNE(n_components=2, random_state=seed, init="pca", perplexity=30).fit_transform(X_sample)
    for clase in sorted(set(y_sample)):
        mask = y_sample == clase
        ax.scatter(emb[mask, 0], emb[mask, 1], s=10, label=clase, alpha=0.7)
    ax.set_title(f"t-SNE (seed={seed})")
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(fontsize=8, frameon=False)
fig.suptitle("Mapas t-SNE con dos semillas -- solo variables numericas (anio, cantidad_denuncias)")
fig.tight_layout()

fig.savefig(REPORTS_DIR / "tsne_two_seeds.png", dpi=150)
plt.show()

print(
    "Nota: el bloque numerico de este dataset solo tiene 2 columnas "
    "(anio, cantidad_denuncias), asi que t-SNE parte de un espacio ya muy "
    "chico. Compara ambos paneles: si la forma general de los grupos se "
    "mantiene entre semillas, es una señal de estabilidad; si cambia "
    "mucho, la estructura visual no es confiable con estas variables."
)

## Paso 6: Calcular la frontera de Pareto

In [ ]:
from inf8239_u01.green import pareto_flags

# Se agrega tambien la fila de svm_c1_pca a la tabla de comparacion.
row_pca = {
    "model": "svm_c1_pca",
    "f1_macro": f1_pca,
    "fit_median_s": fit_s_pca,
    "predict_ms": predict_ms_pca,
    "size_kb": size_kb_pca,
}
results_full = pd.concat([results, pd.DataFrame([row_pca])], ignore_index=True)

results_full["es_pareto"] = pareto_flags(results_full, score="f1_macro", cost="fit_median_s")
results_full = results_full.sort_values(["es_pareto", "f1_macro"], ascending=[False, False]).reset_index(drop=True)
display(results_full)

results_full.to_csv(REPORTS_DIR / "green_ai_results.csv", index=False)
print("Actualizado:", REPORTS_DIR / "green_ai_results.csv")

## Paso 7: Graficar y cuantificar la decisión

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=150)

pareto_pts = results_full[results_full["es_pareto"]]
dominado_pts = results_full[~results_full["es_pareto"]]

ax.scatter(dominado_pts["fit_median_s"], dominado_pts["f1_macro"],
           color="#898781", s=70, label="Dominado", zorder=3)
ax.scatter(pareto_pts["fit_median_s"], pareto_pts["f1_macro"],
           color="#2a78d6", s=90, label="Frontera de Pareto", zorder=4)

for _, row in results_full.iterrows():
    ax.annotate(row["model"], (row["fit_median_s"], row["f1_macro"]),
                xytext=(5, 5), textcoords="offset points", fontsize=8)

ax.set_xlabel("Tiempo de entrenamiento (mediana de 3, segundos)")
ax.set_ylabel("F1-macro (test)")
ax.set_title("Costo computacional vs. desempeño -- frontera de Pareto", loc="left")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(REPORTS_DIR / "pareto.png", dpi=150)
plt.show()

### Paso 7 (continuación): Decisión razonada — completar después de ejecutar

> Esta celda es una plantilla. Después de correr el notebook, reemplaza cada
> `___` con los valores reales de la tabla `results_full` y del gráfico
> `reports/pareto.png`, y desarrolla el párrafo hasta 300-500 palabras.

- **Modelo con mayor F1-macro:** ___ (F1 = ___)
- **Alternativa elegida (si no es la de mayor F1):** ___ (F1 = ___)
- **Diferencia absoluta de F1 entre ambas:** ___
- **Ahorro de tiempo de entrenamiento de la alternativa vs. la de mayor F1 (%):** ___
- **Diferencia de tamaño en disco (KB):** ___
- **Limitaciones de esta medición:** recordar que son tiempos de esta
  máquina en este momento, no una medición de energía/CO2; variables como
  número de núcleos, carga del sistema y versión de scikit-learn afectan
  el resultado.
- **Contexto de hardware:** completar con la salida de la celda de la
  sección "Paso 9: Registrar el entorno" (plataforma, versión de Python,
  procesador, versión de scikit-learn).

**Redacción final (300-500 palabras):**

*(escribir aquí el párrafo de decisión razonada, integrando los puntos de
arriba en prosa)*

## Paso 8: Probar la frontera de Pareto

Casos de prueba para `pareto_flags` en `tests/test_green.py` (se corren
fuera de este notebook, con `pytest` desde la raíz del proyecto):

```bash
pytest tests/test_green.py -v
```

Cubren dos casos: (1) un modelo claramente dominado en ambos ejes queda
excluido de la frontera, y (2) dos modelos empatados en score y costo
permanecen ambos en la frontera, porque ninguno domina estrictamente al
otro.

## Paso 9: Registrar el entorno

In [ ]:
import platform
import sklearn

print("Plataforma:", platform.platform())
print("Version de Python:", sys.version)
print("Procesador:", platform.processor())
print("Version de scikit-learn:", sklearn.__version__)

print(
    "\nNota: estos tiempos se midieron en esta maquina, en este momento, "
    "bajo la carga que tuviera el sistema en ese instante. No son una "
    "medicion de energia ni de CO2 -- para eso haria falta un medidor de "
    "consumo especifico, conocer la region/fuente electrica y documentar "
    "esos supuestos explicitamente. Aqui 'Green AI' se limita a comparar "
    "costo computacional relativo (tiempo, tamaño en disco) entre modelos, "
    "no a cuantificar huella de carbono real."
)